# 2. Calcul des délais d'accès

Ce notebook part des données préparées par le notebook 1 et :
1. trouve la caserne la plus proche de chaque forêt ;
2. convertit la distance en délai ;
3. rattache chaque forêt à sa commune ;
4. calcule les indicateurs par commune et la typologie ;
5. enregistre les résultats pour le notebook 3.

**Hypothèse :** délai (min) = distance à vol d'oiseau (km) × 1,3 / 50 km/h × 60. C'est un **délai théorique minimal de trajet**, sans le temps de traitement de l'alerte ni le temps de départ des équipes.

## 0. Mise en place

In [ ]:
import os
import sys

# je cherche la racine du projet (le dossier qui contient "modules")
racine_projet = None
for essai in [".", "..", "../.."]:
    if os.path.exists(os.path.join(essai, "modules")) and racine_projet is None:
        racine_projet = os.path.abspath(essai)
print("Racine du projet :", racine_projet)

# j'ajoute la racine aux endroits où Python cherche les modules
if racine_projet not in sys.path:
    sys.path.append(racine_projet)

# si je modifie un module, il est rechargé automatiquement
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import geopandas as gpd

from modules.intern.ats import parametres as p
from modules.intern.ats import calculs

## 1. Lecture des données préparées

In [ ]:
fichier_prep = "sorties/donnees_preparees.gpkg"
communes = gpd.read_file(fichier_prep, layer="communes")
casernes = gpd.read_file(fichier_prep, layer="casernes")
points_forets = gpd.read_file(fichier_prep, layer="points_forets")
nb_incendies = pd.read_csv("sorties/nb_incendies_par_commune.csv", sep=";", dtype={"Code INSEE": str})

print("Communes :", len(communes), "| Casernes :", len(casernes), "| Forêts :", len(points_forets))

## 2. Caserne la plus proche et délai

In [ ]:
resultat = calculs.caserne_plus_proche(points_forets, casernes)
resultat = calculs.distance_vers_delai(resultat, p.COEF_DETOUR, p.VITESSE_KMH)
resultat[["id_foret", "nom_caserne", "distance_km", "delai_min"]].head()

## 3. Rattachement aux communes et indicateurs

In [ ]:
forets_communes = calculs.rattacher_aux_communes(resultat, communes, p.COL_CODE, p.COL_NOM, p.COL_DEP)
forets_communes = calculs.classer_delais(forets_communes, p.BORNES_DELAI, p.NOMS_CLASSES_DELAI)
print("Forêts situées en PACA :", len(forets_communes))

carte_communes = calculs.statistiques_par_commune(forets_communes, communes, nb_incendies, p.COL_CODE)
carte_communes, seuil_incendies, seuil_delai = calculs.typologie_communes(carte_communes)
print("Seuils (médianes) : incendies =", seuil_incendies, "| délai =", round(seuil_delai, 1), "min")
print(carte_communes["typologie"].value_counts())

In [ ]:
carte_communes[[p.COL_NOM, "nb_forets", "delai_moyen", "nb_incendies", "typologie"]].head()

## 4. Enregistrement des résultats

In [ ]:
fichier_res = "sorties/resultats.gpkg"
forets_communes.to_file(fichier_res, layer="forets_communes", driver="GPKG")
carte_communes.to_file(fichier_res, layer="carte_communes", driver="GPKG")
print("Résultats enregistrés !")